# Frozen validation statistics and visual simulation
Purpose: compare six saved final learned-kernel JEPA models with their paired ViT controls on a newly registered development reserve. Inputs: completed Notebook 12 frozen predictions and immutable protocol; outputs: analysis_results.json, seven PNG figures, CSV evidence and a self-contained HTML view under the run output. Runtime: about 1–3 CPU minutes, no GPU training. Resume: after all frozen prediction arms complete, import this ahr-export cell and call build_analysis(root, out) again; this only rebuilds derived reports and never changes a prediction or model.
The one primary test uses six independent training pairs, not image count as independent training replicas. Power curves explicitly simulate hypothetical advantages; they cannot establish superiority.

In [ ]:
"""Frozen six-seed validation and explicitly hypothetical power simulation.

No training, probe selection, repeated stopping, or official-test evaluation is performed.
Fresh predictions must first be completed under a registered protocol.
"""
from pathlib import Path
import base64, csv, hashlib, html, itertools, json, math, os, time
import numpy as np
from scipy.stats import t as student_t, nct, chi2, binomtest
import matplotlib
if "__file__" in globals():
    _ANALYSIS_SOURCE12 = Path(__file__).resolve()
else:
    _ANALYSIS_SOURCE12 = next((base / suffix for base in [Path.cwd(), *Path.cwd().parents]
        for suffix in ["experiments/12_visual_test/analysis.ipynb", "fmm_jeppa/experiments/12_visual_test/analysis.ipynb"]
        if (base / suffix).exists()), None)
    if _ANALYSIS_SOURCE12 is None:
        raise RuntimeError("Run this notebook from the fmm_jeppa project or its workspace")
os.environ["MPLCONFIGDIR"] = str(_ANALYSIS_SOURCE12.parents[2] / ".cache/matplotlib12")
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

ANALYSIS_SEED = 2026141201
POWER_TRIALS = 10000
BOOTSTRAP_TRIALS = 2000
VARIANTS = ("vit", "learned_kernel")
COLORS = {"vit": "#2D6698", "learned_kernel": "#A94F83", "reused": "#8C9297", "fresh": "#2D6698"}


def _sha12(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _load12(path):
    return json.loads(Path(path).read_text())


def _write12(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w") as stream:
        json.dump(value, stream, indent=2, sort_keys=True, allow_nan=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def paired_seed_test(deltas_pp, alpha=0.05):
    """Six independent training pairs; the image count is not the test's n."""
    values = np.asarray(deltas_pp, dtype=np.float64)
    if values.ndim != 1 or len(values) < 2 or not np.isfinite(values).all():
        raise ValueError("Finite paired seed differences are required")
    count = len(values)
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    se = sd / math.sqrt(count)
    if se == 0:
        t_stat = None
        p_value = 1.0 if mean == 0 else 0.0
        ci = [mean, mean]
        degenerate = True
    else:
        t_stat = mean / se
        p_value = float(2 * student_t.sf(abs(t_stat), count - 1))
        radius = float(student_t.ppf(1 - alpha / 2, count - 1) * se)
        ci = [mean - radius, mean + radius]
        degenerate = False
    sign_vectors = np.asarray(list(itertools.product((-1, 1), repeat=count)), dtype=np.int8)
    signflip_means = (sign_vectors * values[None, :]).mean(axis=1)
    tolerance = max(1, abs(mean)) * 1e-12
    exact_p = float(np.mean(np.abs(signflip_means) >= abs(mean) - tolerance))
    nonzero = int(np.count_nonzero(values))
    positive = int(np.count_nonzero(values > 0))
    negative = int(np.count_nonzero(values < 0))
    sign_p = float(binomtest(positive, nonzero, 0.5).pvalue) if nonzero else 1.0
    return {
        "n_independent_training_pairs": count, "df": count - 1,
        "mean_difference_pp": mean, "sd_difference_pp": sd, "se_difference_pp": se,
        "ci95_pp": ci, "t_statistic": t_stat, "two_sided_p_value": p_value,
        "alpha": alpha, "degenerate_variance": degenerate,
        "primary_positive_superiority": bool(not degenerate and mean > 0 and ci[0] > 0 and p_value < alpha),
        "exact_signflip_p_value": exact_p, "signflip_enumerations": len(sign_vectors),
        "signflip_means_pp": signflip_means.tolist(), "sign_test_p_value": sign_p,
        "positive_pairs": positive, "negative_pairs": negative, "zero_pairs": count - nonzero,
        "paired_differences_pp": values.tolist(),
        "primary_assumptions": "Independent paired training seeds; approximately normal paired seed effects. n=6 makes this assumption difficult to assess.",
        "signflip_assumptions": "Pairwise exchangeability under the null, or symmetric independent paired differences. This is a robustness check, not a second chance to choose significance.",
        "scope": "Conditional on these fixed encoders, fixed probe rule, Food-101 split and bounded 1,800-update JEPA protocol; not a general architecture theorem."
    }


def crossed_bootstrap(correct, labels, trials=BOOTSTRAP_TRIALS, seed=ANALYSIS_SEED):
    """Resample paired seeds and images within class, sharing image draws across seeds.

    Fixed class proportions are retained; classes themselves are not sampled.
    This secondary percentile interval is exploratory with only six training pairs.
    """
    correct = np.asarray(correct, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    if correct.ndim != 3 or correct.shape[1] != 2 or correct.shape[2] != len(labels):
        raise ValueError("Expected paired correctness [seed, variant, image]")
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(labels == label) for label in sorted(set(labels.tolist()))]
    difference = correct[:, 1] - correct[:, 0]
    n_seeds, n_images = difference.shape
    draws = np.empty(trials, dtype=np.float64)
    for index in range(trials):
        seed_indices = rng.integers(0, n_seeds, size=n_seeds)
        paired_mean_per_image = difference[seed_indices].mean(axis=0)
        total = 0.0
        for group in groups:
            sampled_images = group[rng.integers(0, len(group), size=len(group))]
            total += float(paired_mean_per_image[sampled_images].sum())
        draws[index] = 100 * total / n_images
    return {
        "method": "crossed seed and image bootstrap, images stratified within fixed classes",
        "trials": trials, "seed": seed, "mean_difference_pp": float(100 * difference.mean()),
        "percentile95_pp": np.quantile(draws, [0.025, 0.975]).tolist(),
        "probability_positive_in_bootstrap": float(np.mean(draws > 0)),
        "warning": "Secondary uncertainty description, not the primary test or a posterior probability. Six seed pairs constrain reliability; classes and dataset membership are fixed. More images do not create more independent training seeds."
    }, draws


def normal_power(delta_pp, sd_pp, count, alpha=0.05):
    """P(positive two-sided rejection), not P(either-direction rejection)."""
    if count < 2 or sd_pp <= 0:
        raise ValueError("At least two pairs and positive population SD required")
    threshold = float(student_t.ppf(1 - alpha / 2, count - 1))
    return float(nct.sf(threshold, count - 1, delta_pp * math.sqrt(count) / sd_pp))


def pairs_for_power(delta_pp, sd_pp, target=0.8, maximum=1000000):
    if delta_pp <= 0:
        return None
    high = 2
    while high < maximum and normal_power(delta_pp, sd_pp, high) < target:
        high = min(maximum, high * 2)
    if normal_power(delta_pp, sd_pp, high) < target:
        return None
    low = 2
    while low < high:
        middle = (low + high) // 2
        if normal_power(delta_pp, sd_pp, middle) >= target:
            high = middle
        else:
            low = middle + 1
    return int(low)


def simulate_power(observed_sd_pp, trials=POWER_TRIALS, seed=ANALYSIS_SEED + 1):
    if not np.isfinite(observed_sd_pp) or observed_sd_pp <= 0:
        raise ValueError("Observed nonzero seed variance required for scenario planning")
    counts = [6, 12, 24, 48, 100, 250, 500, 1000]
    effects = [0.0, 0.1, 0.25, 0.5, 1.0]
    scales = [0.5, 1.0, 2.0]
    rng = np.random.default_rng(seed)
    rows = []
    null_rows = []
    for count in counts:
        z_mean = rng.standard_normal(trials)
        variance_scale = np.sqrt(rng.chisquare(count - 1, size=trials) / (count - 1))
        threshold = float(student_t.ppf(0.975, count - 1))
        null_statistics = z_mean / variance_scale
        both = float(np.mean(np.abs(null_statistics) > threshold))
        positive = float(np.mean(null_statistics > threshold))
        null_rows.append({
            "pairs": count, "two_sided_null_rejection_rate": both,
            "expected_two_sided_null_rate": 0.05,
            "positive_superiority_null_rate": positive,
            "expected_positive_superiority_null_rate": 0.025,
            "monte_carlo_standard_error_two_sided": math.sqrt(0.05 * 0.95 / trials)
        })
        for scale in scales:
            sd = scale * observed_sd_pp
            for effect in effects:
                simulated_t = (z_mean + effect * math.sqrt(count) / sd) / variance_scale
                empirical = float(np.mean(simulated_t > threshold))
                analytical = normal_power(effect, sd, count)
                rows.append({
                    "pairs": count, "assumed_true_effect_pp": effect,
                    "sd_sensitivity_multiplier": scale, "assumed_population_sd_pp": sd,
                    "analytical_positive_rejection_power": analytical,
                    "simulated_positive_rejection_power": empirical,
                    "monte_carlo_trials": trials,
                    "monte_carlo_standard_error": math.sqrt(max(analytical * (1 - analytical), 1 / trials) / trials)
                })
    requirements = [{
        "assumed_true_effect_pp": effect, "sd_sensitivity_multiplier": scale,
        "assumed_population_sd_pp": scale * observed_sd_pp,
        "pairs_for_80_percent_power": pairs_for_power(effect, scale * observed_sd_pp)
    } for effect in effects if effect > 0 for scale in scales]
    sigma_interval = [
        math.sqrt(5 * observed_sd_pp ** 2 / chi2.ppf(0.975, 5)),
        math.sqrt(5 * observed_sd_pp ** 2 / chi2.ppf(0.025, 5))
    ]
    return {
        "normal_paired_difference_scenarios": rows,
        "requirements_80_percent": requirements,
        "null_calibration": null_rows,
        "observed_development_sd_pp": observed_sd_pp,
        "normal_theory_sd_ci95_from_six_development_pairs_pp": sigma_interval,
        "trials_per_scenario": trials, "seed": seed,
        "decision_rule": "two-sided p < .05 and positive mean; equivalent to positive t > t_.975",
        "interpretation": "Simulation explores assumed effects, not observed victories. It cannot establish that learned kernel beats ViT. The reused development SD estimates variance from only six pairs; true variance may differ. More image evaluations do not replace new independently trained seed pairs."
    }


def _read_development12(root, seeds):
    stage = Path(root) / "artifacts/notebook_11/manifold-long-001/stages/learning"
    sources = []
    predictions = []
    sample_ids = None
    labels = None
    for seed in seeds:
        arms = []
        for variant in VARIANTS:
            model_id = f"s{seed}-{variant}-u1800"
            evaluation_path = stage / "evaluation" / (model_id + ".json")
            saved = _load12(evaluation_path)
            summary = saved["probe"]["summary"]
            if int(summary["completed_epochs"]) != 20 or summary["selection"] != "fixed_final_epoch":
                raise ValueError("Development baseline probe rule differs")
            path = stage / "predictions" / (model_id + ".csv")
            if not path.exists():
                path = Path(summary["predictions_path"])
            with path.open() as stream:
                rows = list(csv.DictReader(stream))
            ids_here = [row["sample_id"] for row in rows]
            labels_here = np.asarray([int(row["label"]) for row in rows], dtype=np.int64)
            prediction_here = np.asarray([int(row["top1_prediction"]) for row in rows], dtype=np.int64)
            if sample_ids is None:
                sample_ids, labels = ids_here, labels_here
            elif sample_ids != ids_here or not np.array_equal(labels, labels_here):
                raise ValueError("Development samples do not pair across models")
            accuracy = float(np.mean(prediction_here == labels_here))
            if abs(accuracy - float(summary["metrics"]["top1"])) > 1e-6:
                raise ValueError("Development predictions do not reconcile with recorded top1")
            sources.extend([{"path": str(evaluation_path), "sha256": _sha12(evaluation_path)},
                            {"path": str(path), "sha256": _sha12(path)}])
            arms.append(prediction_here)
        predictions.append(arms)
    return np.asarray(predictions), np.asarray(labels), sample_ids, sources


def _read_fresh12(root, out):
    out = Path(out)
    metadata = _load12(out / "predictions.json")
    if metadata.get("status") != "COMPLETE" or metadata.get("fresh_status") != "COMPLETE":
        raise ValueError("Fresh validation must be complete before analysis")
    npz_path = Path(metadata["npz_path"])
    if not npz_path.resolve().is_relative_to(Path(root).resolve()) or _sha12(npz_path) != metadata["npz_sha256"]:
        raise ValueError("Fresh predictions path or hash is invalid")
    if tuple(metadata["variants"]) != VARIANTS:
        raise ValueError("Prespecified comparison variant order differs")
    seeds = [int(value) for value in metadata["seeds"]]
    if seeds != list(range(20261401, 20261407)):
        raise ValueError("All six registered independent training pairs are required")
    with np.load(npz_path, allow_pickle=False) as archive:
        predictions = archive["predictions"]
        labels = archive["labels"]
        ids = archive["sample_ids"].tolist()
    if predictions.shape != (6, 2, len(labels)) or len(ids) != len(labels):
        raise ValueError("Fresh prediction dimensions are invalid")
    if ids != list(metadata["sample_ids"]) or not np.array_equal(labels, metadata["labels"]):
        raise ValueError("Fresh metadata does not reconcile with the NPZ")
    if len(set(ids)) != len(ids) or set(labels.tolist()) != set(range(101)):
        raise ValueError("Unique image identities and all 101 classes are required")
    if np.any(predictions < 0) or np.any(predictions >= 101):
        raise ValueError("Prediction labels are outside the Food101 class mapping")
    protocol_path = Path(metadata.get("protocol_path", out / "protocol.json"))
    if _sha12(protocol_path) != metadata["registration_sha256"]:
        raise ValueError("Registered protocol changed before analysis")
    return metadata, predictions, labels, ids, protocol_path


def _save_figure12(fig, folder, name, caption, evidence, layout_hpad=2.0):
    fig.tight_layout(h_pad=layout_hpad, pad=1.5)
    path = Path(folder) / name
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return {"path": str(path), "sha256": _sha12(path), "caption": caption, "evidence": evidence}


def _gallery12(root, folder, predictions, labels, ids, names, seed):
    from fmm_jepa.food101_data import Food101Adapter
    root = Path(root)
    dataset = Food101Adapter(root / "data/food101", _load12(root / "data/food101/dataset_manifest.json"))
    correct = predictions[0] == labels[None, :]
    groups = [
        ("Both correct", correct[0] & correct[1]),
        ("ViT only correct", correct[0] & ~correct[1]),
        ("Learned kernel only correct", ~correct[0] & correct[1]),
        ("Both incorrect", ~correct[0] & ~correct[1])
    ]
    fig, axes = plt.subplots(4, 3, figsize=(12, 18))
    selected = []
    for row_index, (group_name, mask) in enumerate(groups):
        candidates = sorted(np.flatnonzero(mask).tolist(), key=lambda index: ids[index])
        for column_index in range(3):
            axis = axes[row_index, column_index]
            axis.set_axis_off()
            if column_index >= len(candidates):
                axis.text(0.5, 0.5, "No remaining example", ha="center", va="center")
                continue
            index = candidates[column_index]
            record = dataset.get(ids[index])
            if record["label"] != int(labels[index]):
                raise ValueError("Gallery image source label differs")
            image = ImageOps.fit(record["image"], (280, 210))
            axis.imshow(image)
            actual = names[int(labels[index])].replace("_", " ")
            vit = names[int(predictions[0, 0, index])].replace("_", " ")
            kernel = names[int(predictions[0, 1, index])].replace("_", " ")
            axis.set_title(f"{group_name}\nTrue: {actual}\nViT: {vit}\nKernel: {kernel}", fontsize=10)
            source_path = dataset.path_for(ids[index])
            selected.append({"group": group_name, "sample_id": ids[index], "label": int(labels[index]),
                             "vit_prediction": int(predictions[0, 0, index]),
                             "learned_prediction": int(predictions[0, 1, index]),
                             "image_path": str(source_path), "image_sha256": _sha12(source_path)})
    fig.suptitle(f"Illustrative image outcomes — first registered seed {seed}", fontsize=15, y=1.003)
    return _save_figure12(fig, folder, "image_outcomes.png",
        "First registered seed, first three lexicographic IDs in each outcome group. These illustrative images cannot determine statistical significance. Images are center-cropped only for display.",
        {"selection": "first seed; first three sorted IDs within each fixed outcome group", "images": selected}, layout_hpad=6.0)


def _csv12(path, rows):
    if not rows:
        return
    with Path(path).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def register_analysis(root, out):
    """Freeze this analysis before observing any fresh-reserve predictions."""
    root, out = Path(root).resolve(), Path(out).resolve()
    if not out.is_relative_to(root):
        raise ValueError("Analysis registration must belong under this project")
    source_path = _ANALYSIS_SOURCE12
    protocol_path = out / "protocol.json"
    if not protocol_path.exists():
        raise ValueError("Frozen evaluation must be registered before its analysis")
    registration = {
        "schema": "fmm-jepa-frozen-analysis/v1",
        "evaluation_registration_sha256": _sha12(protocol_path),
        "analysis_notebook": str(source_path), "analysis_notebook_sha256": _sha12(source_path),
        "primary_endpoint": "mean final learned-kernel minus ViT top1 accuracy in percentage points across six paired training seeds on all fresh reserved development images",
        "training_seed_ids": list(range(20261401, 20261407)),
        "primary_test": "single two-sided paired t test, df=5, alpha=.05",
        "positive_superiority_rule": "positive paired mean AND two-sided p < .05 AND lower 95% paired-seed t confidence bound > 0",
        "secondary_only": ["64 exact paired sign flips", "sign test", "class differences", "image outcomes", "crossed bootstrap", "power simulation"],
        "bootstrap_trials": BOOTSTRAP_TRIALS, "bootstrap_seed": ANALYSIS_SEED,
        "power_trials": POWER_TRIALS, "power_seed": ANALYSIS_SEED + 1,
        "power_assumed_effects_pp": [0.0, 0.1, 0.25, 0.5, 1.0],
        "power_seed_pair_counts": [6, 12, 24, 48, 100, 250, 500, 1000],
        "power_sd_multipliers": [0.5, 1.0, 2.0],
        "power_sd_source": "six reused Notebook11 development paired deltas, no fresh outcome used for power planning",
        "gallery_selection": "first registered training seed; first three lexicographically sorted sample IDs in each of both-correct, ViT-only-correct, kernel-only-correct, both-incorrect",
        "multiple_testing": "one primary comparison; no class tests, quality selection, interim significance testing or optional stopping",
        "official_test": "NOT_RUN", "fresh_predictions_observed_before_registration": False
    }
    path = out / "analysis_protocol.json"
    if path.exists():
        if _load12(path) != registration:
            raise ValueError("Immutable analysis registration differs; never silently replace it")
    else:
        arm_outputs = out / "predictions"
        if (out / "predictions.json").exists() or (out / "predictions.npz").exists() or (arm_outputs.exists() and any(arm_outputs.glob("*.csv"))):
            raise ValueError("Cannot newly preregister analysis after fresh predictions exist")
        _write12(path, registration)
    return {"status": "REGISTERED", "path": str(path), "sha256": _sha12(path), "analysis_notebook_sha256": registration["analysis_notebook_sha256"]}


def build_analysis(root, out):
    """Read completed frozen validation; save six-seed statistics, simulation and visuals."""
    root, out = Path(root).resolve(), Path(out).resolve()
    started = time.time()
    analysis_registration = _load12(out / "analysis_protocol.json")
    if analysis_registration.get("analysis_notebook_sha256") != _sha12(_ANALYSIS_SOURCE12):
        raise ValueError("Analysis code changed after immutable preregistration")
    if analysis_registration.get("evaluation_registration_sha256") != _sha12(out / "protocol.json"):
        raise ValueError("Evaluation registration changed after analysis preregistration")
    if not out.is_relative_to(root):
        raise ValueError("Analysis output must belong under this project")
    metadata, fresh_predictions, labels, ids, protocol_path = _read_fresh12(root, out)
    seeds = [int(value) for value in metadata["seeds"]]
    names = metadata["class_names"]
    if len(names) != 101:
        raise ValueError("Class mapping differs")
    development_predictions, dev_labels, dev_ids, sources = _read_development12(root, seeds)
    if set(ids) & set(dev_ids):
        raise ValueError("Fresh validation overlaps the previously reported development subset")
    split_path = root / "artifacts/notebook_11/manifold-long-001/stages/learning/split_manifest.json"
    split = _load12(split_path)
    expected = set(split["pool_ids"]["development"]) - set(split["selected_ids"]["development"])
    if set(ids) != expected or set(ids) & set(split["selected_ids"]["training"]) or set(ids) & set(split["selected_ids"]["heldout"]):
        raise ValueError("Fresh validation membership differs from the reserved development complement")
    for index, seed in enumerate(seeds):
        for variant_index, variant in enumerate(VARIANTS):
            arm = next((row for row in metadata["arms"] if int(row["seed"]) == seed and row["variant"] == variant), None)
            measured = 100 * np.mean(fresh_predictions[index, variant_index] == labels)
            if arm is None or abs(measured - float(arm["top1_percent"])) > 1e-6:
                raise ValueError("Fresh arm summary does not reconcile with predictions")
    correct = fresh_predictions == labels[None, None, :]
    dev_correct = development_predictions == dev_labels[None, None, :]
    fresh_accuracy = 100 * correct.mean(axis=2)
    dev_accuracy = 100 * dev_correct.mean(axis=2)
    fresh_test = paired_seed_test(fresh_accuracy[:, 1] - fresh_accuracy[:, 0])
    development_test = paired_seed_test(dev_accuracy[:, 1] - dev_accuracy[:, 0])
    bootstrap, bootstrap_draws = crossed_bootstrap(correct, labels)
    power = simulate_power(development_test["sd_difference_pp"])
    figures_dir = out / "visual_report"
    figures_dir.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.labelcolor": "#263238", "text.color": "#263238"})
    figures = []
    seed_rows = []
    for index, seed in enumerate(seeds):
        seed_rows.append({"seed": seed, "fresh_vit_top1_percent": float(fresh_accuracy[index, 0]),
            "fresh_learned_kernel_top1_percent": float(fresh_accuracy[index, 1]),
            "fresh_difference_pp": float(fresh_accuracy[index, 1] - fresh_accuracy[index, 0]),
            "reused_dev_vit_top1_percent": float(dev_accuracy[index, 0]),
            "reused_dev_learned_kernel_top1_percent": float(dev_accuracy[index, 1]),
            "reused_dev_difference_pp": float(dev_accuracy[index, 1] - dev_accuracy[index, 0])})
    _csv12(figures_dir / "paired_seed_scores.csv", seed_rows)
    _csv12(figures_dir / "power_scenarios.csv", power["normal_paired_difference_scenarios"])
    _csv12(figures_dir / "power_requirements.csv", power["requirements_80_percent"])
    _csv12(figures_dir / "null_calibration.csv", power["null_calibration"])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    y = np.arange(6)
    for index in range(6):
        axes[0].plot(fresh_accuracy[index], [index, index], color="#AAB1B7", linewidth=1.5)
    axes[0].scatter(fresh_accuracy[:, 0], y, color=COLORS["vit"], s=55, marker="o", label="ViT", zorder=3)
    axes[0].scatter(fresh_accuracy[:, 1], y, color=COLORS["learned_kernel"], s=55, marker="D", label="Learned kernel", zorder=3)
    axes[0].set(yticks=y, yticklabels=[str(seed) for seed in seeds], xlabel="Fresh top-1 accuracy (%)", title="Fresh reserve: paired model scores")
    axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
    axes[0].grid(axis="x", color="#E3E7E9", zorder=0)
    axes[1].scatter(dev_accuracy[:, 1] - dev_accuracy[:, 0], y - 0.12, color=COLORS["reused"], marker="o", s=45, label="Reused development (exploratory)")
    axes[1].scatter(fresh_accuracy[:, 1] - fresh_accuracy[:, 0], y + 0.12, color=COLORS["fresh"], marker="D", s=45, label="Fresh reserved development")
    axes[1].axvline(0, color="#263238", linewidth=1)
    axes[1].set(yticks=y, yticklabels=[str(seed) for seed in seeds], xlabel="Learned kernel − ViT (percentage points)", title="Six training pairs, two image subsets")
    axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), fontsize=9, frameon=False)
    figures.append(_save_figure12(fig, figures_dir, "paired_seed_scores.png",
        f"All six paired seeds; each fresh score evaluates the same {len(ids):,} reserved images. The old 2,020-image subset is reused exploratory evidence. No seed selection.", seed_rows))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    tests = [("Fresh: primary paired t", fresh_test, COLORS["fresh"]),
             ("Reused dev: exploratory", development_test, COLORS["reused"])]
    for index, (label, result, color) in enumerate(tests):
        mean = result["mean_difference_pp"]
        low, high = result["ci95_pp"]
        axes[0].errorbar(mean, index, xerr=[[mean - low], [high - mean]], fmt="D", color=color, capsize=5, markersize=7)
    axes[0].axvline(0, color="#263238", linewidth=1)
    axes[0].set(yticks=[0, 1], yticklabels=[row[0] for row in tests], xlabel="Mean learned kernel − ViT (pp)", title="95% paired-seed t intervals; n = 6")
    null = np.asarray(fresh_test["signflip_means_pp"])
    axes[1].hist(null, bins=14, color="#8BAAC1", edgecolor="white")
    axes[1].axvline(fresh_test["mean_difference_pp"], color=COLORS["learned_kernel"], linewidth=2, label="Observed fresh mean")
    axes[1].axvline(-fresh_test["mean_difference_pp"], color=COLORS["learned_kernel"], linestyle="--", linewidth=1)
    axes[1].set(xlabel="Signed paired mean difference (pp)", ylabel="Number of sign assignments", title=f"All 64 sign flips; two-sided p = {fresh_test['exact_signflip_p_value']:.4f}")
    axes[1].legend(frameon=False, fontsize=9)
    figures.append(_save_figure12(fig, figures_dir, "primary_interval_and_signflips.png",
        "Primary: one predeclared two-sided paired t test at α=.05, with positive mean and lower 95% bound above zero. Sign flips are a robustness check under exchangeability/symmetry, not an alternative selected after seeing results.",
        {"fresh": fresh_test, "reused_development": development_test}))

    fig, axis = plt.subplots(figsize=(10, 4))
    axis.hist(bootstrap_draws, bins=45, color="#8BAAC1", edgecolor="white")
    axis.axvline(0, color="#263238", linewidth=1)
    for value in bootstrap["percentile95_pp"]:
        axis.axvline(value, color=COLORS["learned_kernel"], linestyle="--", linewidth=1.5)
    axis.set(xlabel="Resampled learned kernel − ViT (pp)", ylabel="Bootstrap draws", title="Secondary: crossed seed + within-class image bootstrap")
    figures.append(_save_figure12(fig, figures_dir, "crossed_bootstrap.png",
        "2,000 paired draws resample six seeds and image IDs within fixed classes, with the same image draw shared across seeds. Percentile intervals are secondary and limited by n=6; bootstrap mass above zero is not a superiority probability.", bootstrap))

    class_rows = []
    class_effects = np.zeros((101, 6))
    for label in range(101):
        member = labels == label
        class_effects[label] = 100 * (correct[:, 1, member].mean(axis=1) - correct[:, 0, member].mean(axis=1))
        class_rows.append({"class_label": label, "class_name": names[label], "images": int(member.sum()),
                           "mean_difference_pp": float(class_effects[label].mean()),
                           **{f"seed_{seed}_difference_pp": float(class_effects[label, index]) for index, seed in enumerate(seeds)}})
    _csv12(figures_dir / "class_differences.csv", class_rows)
    fig, axes = plt.subplots(1, 3, figsize=(15, 13), sharex=True)
    limit = max(1, float(np.max(np.abs(class_effects))))
    blocks = np.array_split(np.arange(101), 3)
    for axis, block in zip(axes, blocks):
        mesh = axis.imshow(class_effects[block], cmap="PuOr", vmin=-limit, vmax=limit, aspect="auto")
        axis.set(yticks=np.arange(len(block)), yticklabels=[names[label].replace("_", " ") for label in block],
                 xticks=np.arange(6), xticklabels=[str(seed)[-2:] for seed in seeds], xlabel="Seed suffix")
        axis.tick_params(axis="y", labelsize=9)
    fig.suptitle("Descriptive class differences: learned kernel − ViT (pp)", fontsize=15, y=1.005)
    fig.subplots_adjust(right=0.91, wspace=0.7)
    color_axis = fig.add_axes([0.93, 0.12, 0.015, 0.76])
    fig.colorbar(mesh, cax=color_axis, label="Percentage points")
    # Keep the manually allocated colorbar clear of tight-layout repositioning.
    path = figures_dir / "class_differences.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    figures.append({"path": str(path), "sha256": _sha12(path), "caption": "All 101 classes in source order, all six seeds. Descriptive only: no 101 per-class significance tests or class selection. Unequal class counts are retained in the CSV.", "evidence": class_rows})

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    scenario_colors = {0.1: "#2D6698", 0.25: "#B8791B", 0.5: "#A94F83", 1.0: "#6C7835"}
    rows = power["normal_paired_difference_scenarios"]
    for effect in [0.1, 0.25, 0.5, 1.0]:
        selected = [row for row in rows if row["assumed_true_effect_pp"] == effect and row["sd_sensitivity_multiplier"] == 1]
        axes[0].plot([row["pairs"] for row in selected], [row["analytical_positive_rejection_power"] for row in selected],
                     color=scenario_colors[effect], marker="o", label=f"Assumed +{effect:g} pp")
        axes[0].scatter([row["pairs"] for row in selected], [row["simulated_positive_rejection_power"] for row in selected],
                        facecolors="none", edgecolors=scenario_colors[effect], s=55)
    axes[0].axhline(0.8, color="#263238", linewidth=1, linestyle="--")
    axes[0].set(xscale="log", ylim=(0, 1.02), xlabel="New independently trained seed pairs", ylabel="Probability of positive significant result", title="Hypothetical power; observed development SD")
    axes[0].legend(frameon=False, fontsize=9)
    styles = {0.5: "--", 1.0: "-", 2.0: ":"}
    for scale in [0.5, 1.0, 2.0]:
        selected = [row for row in rows if row["assumed_true_effect_pp"] == 0.25 and row["sd_sensitivity_multiplier"] == scale]
        axes[1].plot([row["pairs"] for row in selected], [row["analytical_positive_rejection_power"] for row in selected],
                     color=COLORS["fresh"], linestyle=styles[scale], marker="o", label=f"SD × {scale:g}")
    axes[1].axhline(0.8, color="#263238", linewidth=1, linestyle="--")
    axes[1].set(xscale="log", ylim=(0, 1.02), xlabel="New independently trained seed pairs", ylabel="Probability of positive significant result", title="Variance sensitivity; assumed +0.25 pp")
    axes[1].legend(frameon=False)
    figures.append(_save_figure12(fig, figures_dir, "hypothetical_power.png",
        f"Normal paired-effect scenarios, not evidence of an actual win. Lines are exact noncentral-t power; open circles are {POWER_TRIALS:,}-trial simulation. Six reused development pairs supply a noisy SD estimate of {development_test['sd_difference_pp']:.3f} pp.", power))

    fig, axis = plt.subplots(figsize=(10, 4))
    null_rows = power["null_calibration"]
    counts = [row["pairs"] for row in null_rows]
    axis.plot(counts, [row["two_sided_null_rejection_rate"] for row in null_rows], "o-", color=COLORS["vit"], label="Either direction: expected 5%")
    axis.plot(counts, [row["positive_superiority_null_rate"] for row in null_rows], "D--", color=COLORS["learned_kernel"], label="Positive superiority rule: expected 2.5%")
    axis.axhline(0.05, color=COLORS["vit"], alpha=0.5, linewidth=1)
    axis.axhline(0.025, color=COLORS["learned_kernel"], alpha=0.5, linewidth=1)
    axis.set(xscale="log", ylim=(0, 0.07), xlabel="Independently trained seed pairs in simulation", ylabel="Null rejection frequency", title="Null calibration: no assumed architecture advantage")
    axis.legend(frameon=False)
    figures.append(_save_figure12(fig, figures_dir, "null_calibration.png",
        "Under normal zero-effect paired differences, the two-sided test rejects in either direction at 5%; requiring a positive result halves the false positive rate. Simulation checks implementation, not the model's real-world superiority.", null_rows))
    figures.append(_gallery12(root, figures_dir, fresh_predictions, labels, ids, names, seeds[0]))

    positive = fresh_test["primary_positive_superiority"]
    negative = bool(fresh_test["ci95_pp"][1] < 0 and fresh_test["two_sided_p_value"] < .05)
    verdict = "POSITIVE_SUPERIORITY_ON_FRESH_RESERVED_DEVELOPMENT" if positive else ("VIT_SUPERIOR_ON_FRESH_RESERVED_DEVELOPMENT" if negative else "NO_STATISTICALLY_SUPPORTED_SUPERIORITY")
    decision = {
        "status": "COMPLETE", "verdict": verdict,
        "primary": fresh_test, "fresh_vit_mean_top1_percent": float(fresh_accuracy[:, 0].mean()),
        "fresh_learned_mean_top1_percent": float(fresh_accuracy[:, 1].mean()),
        "fresh_image_count": len(ids), "paired_training_seeds": seeds,
        "reused_development": {"status": "EXPLORATORY_REUSED_DATA", "test": development_test, "image_count": len(dev_ids)},
        "secondary_crossed_bootstrap": bootstrap, "hypothetical_power": power,
        "heldout_official_test_status": "NOT_RUN", "new_training_status": "NOT_RUN",
        "registration_sha256": _sha12(protocol_path), "analysis_registration_sha256": _sha12(out / "analysis_protocol.json"),
        "predictions_sha256": metadata["npz_sha256"],
        "source_evidence": sources + [{"path": str(protocol_path), "sha256": _sha12(protocol_path)},
                                     {"path": str(split_path), "sha256": _sha12(split_path)},
                                     {"path": str(out / "predictions.json"), "sha256": _sha12(out / "predictions.json")}],
        "limitations": [
            "The independent training sample size is six paired seeds, not six times the image count.",
            "The fresh reserved development membership was fixed before viewing these predictions, but architecture design was informed by earlier development results.",
            "The main paired t conclusion depends on approximately normal independent seed effects; n=6 limits assumption checking.",
            "Official heldout test data remains untouched. This is a development confirmation under a bounded custom JEPA protocol.",
            "Image examples, class heatmaps, bootstrap intervals and power simulation are descriptive or secondary; none replaces the prespecified primary test.",
            "No claim of equivalence follows from failure to reject. No FMM acceleration or general model superiority is established."
        ],
        "analysis_runtime_seconds": time.time() - started
    }
    _write12(out / "analysis_results.json", decision)
    _write12(figures_dir / "figure_evidence.json", {"status": "COMPLETE", "figures": figures, "seed_rows": seed_rows})
    mean = fresh_test["mean_difference_pp"]
    low, high = fresh_test["ci95_pp"]
    if positive:
        headline = "Learned kernel beats ViT on the fresh reserved development comparison"
    elif negative:
        headline = "ViT beats learned kernel on the fresh reserved development comparison"
    else:
        headline = "Fresh comparison does not demonstrate that learned kernel beats ViT"
    fragments = ["<!doctype html><html lang='en'><meta charset='utf-8'><meta name='viewport' content='width=device-width, initial-scale=1'>",
        "<title>Frozen JEPA validation and power simulation</title><style>body{font-family:system-ui,sans-serif;color:#263238;background:#fff;max-width:1120px;margin:auto;padding:24px;line-height:1.5}h1{font-size:1.8rem}figure{margin:32px 0}img{width:100%;height:auto}figcaption{color:#495861;font-size:.94rem}table{border-collapse:collapse;width:100%;margin:18px 0}th,td{padding:8px;border-bottom:1px solid #dce2e5;text-align:left}code{overflow-wrap:anywhere}a{color:#2D6698}</style>",
        "<h1>" + html.escape(headline) + "</h1>",
        f"<p>Six paired training seeds, {len(ids):,} fresh reserved Food-101 development images each. Frozen final encoders and fixed epoch-20 probes; no new training, architecture selection or official-test evaluation.</p>",
        f"<p><strong>Learned kernel − ViT: {mean:+.3f} percentage points.</strong> 95% paired-seed t interval [{low:+.3f}, {high:+.3f}]; two-sided p={fresh_test['two_sided_p_value']:.4f}. Sign-flip robustness p={fresh_test['exact_signflip_p_value']:.4f}. {fresh_test['positive_pairs']} positive, {fresh_test['negative_pairs']} negative and {fresh_test['zero_pairs']} tied seeds.</p>",
        "<p>The primary decision was fixed before viewing these fresh predictions: positive effect, two-sided p&lt;.05 and lower 95% paired-seed t bound above zero. Secondary plots and simulations do not change that decision. With only six training pairs, seed-distribution assumptions remain difficult to assess.</p>",
        "<table><tr><th>Encoder</th><th>Mean fresh top-1</th></tr>",
        f"<tr><td>ViT</td><td>{decision['fresh_vit_mean_top1_percent']:.3f}%</td></tr><tr><td>Learned kernel</td><td>{decision['fresh_learned_mean_top1_percent']:.3f}%</td></tr></table>"]
    for figure in figures:
        encoded = base64.b64encode(Path(figure["path"]).read_bytes()).decode()
        fragments.append("<figure><img alt='" + html.escape(Path(figure["path"]).stem.replace("_", " "), quote=True) + "' src='data:image/png;base64," + encoded + "'><figcaption>" + html.escape(figure["caption"]) + "</figcaption></figure>")
    fragments.append("<h2>Power planning is hypothetical</h2><p>Each assumed true effect below is a scenario, not an estimate guaranteed to persist. Normal paired differences and the reused development SD underpin these sample sizes; actual variance and distribution can differ.</p><table><tr><th>Assumed true advantage</th><th>Pairs for 80% power, SD × 0.5</th><th>SD × 1</th><th>SD × 2</th></tr>")
    for effect in [0.1, 0.25, 0.5, 1.0]:
        row = [entry for entry in power["requirements_80_percent"] if entry["assumed_true_effect_pp"] == effect]
        fragments.append(f"<tr><td>+{effect:g} pp</td>" + "".join(f"<td>{entry['pairs_for_80_percent_power']:,}</td>" for entry in row) + "</tr>")
    fragments.append("</table><h2>Scope and evidence</h2><ul>" + "".join("<li>" + html.escape(item) + "</li>" for item in decision["limitations"]) + "</ul>")
    fragments.append("<p>Registered protocol SHA-256: <code>" + decision["registration_sha256"] + "</code><br>Frozen prediction SHA-256: <code>" + decision["predictions_sha256"] + "</code></p>")
    fragments.append("<p>Inspectable files beside this report: paired_seed_scores.csv, class_differences.csv, power_scenarios.csv, power_requirements.csv, null_calibration.csv and figure_evidence.json. Statistical results are saved in ../analysis_results.json.</p></html>")
    report_path = figures_dir / "report.html"
    temporary = report_path.with_name("report.html.partial")
    temporary.write_text("\n".join(fragments))
    os.replace(temporary, report_path)
    manifest = {"status": "PASS", "html": str(report_path), "html_sha256": _sha12(report_path),
                "figures": [{key: value for key, value in figure.items() if key != "evidence"} for figure in figures],
                "results": str(out / "analysis_results.json"), "results_sha256": _sha12(out / "analysis_results.json"),
                "browser_layout_validation": "NOT_RUN", "new_training": "NOT_RUN", "official_test": "NOT_RUN"}
    _write12(out / "analysis_manifest.json", manifest)
    return {"status": "COMPLETE", "verdict": verdict, "results": str(out / "analysis_results.json"),
            "report": str(report_path), "html": str(report_path),
            "figures": [{"path": figure["path"], "title": Path(figure["path"]).stem.replace("_", " ").title(), "note": figure["caption"]} for figure in figures], "primary": fresh_test,
            "analysis_manifest": str(out / "analysis_manifest.json")}


def analysis_smoke(root):
    """CPU-only method verification on the already observed development predictions."""
    from scipy.stats import ttest_rel
    root = Path(root).resolve()
    predictions, labels, ids, evidence = _read_development12(root, list(range(20261401, 20261407)))
    correct = predictions == labels[None, None, :]
    accuracy = 100 * correct.mean(axis=2)
    result = paired_seed_test(accuracy[:, 1] - accuracy[:, 0])
    scipy_check = ttest_rel(accuracy[:, 1], accuracy[:, 0])
    assert abs(result["two_sided_p_value"] - float(scipy_check.pvalue)) < 1e-12
    assert abs(result["t_statistic"] - float(scipy_check.statistic)) < 1e-12
    assert abs(result["mean_difference_pp"] - 0.09900990099009903) < 1e-10
    assert result["signflip_enumerations"] == 64
    assert result["exact_signflip_p_value"] == 62 / 64
    assert not result["primary_positive_superiority"]
    assert not paired_seed_test(np.zeros(6))["primary_positive_superiority"]
    assert paired_seed_test([1, 2, 3, 4, 5, 6])["primary_positive_superiority"]
    assert not paired_seed_test([-1, -2, -3, -4, -5, -6])["primary_positive_superiority"]
    bootstrap, first = crossed_bootstrap(correct, labels, trials=100)
    _, repeated = crossed_bootstrap(correct, labels, trials=100)
    assert np.array_equal(first, repeated)
    power = simulate_power(result["sd_difference_pp"])
    rows = power["normal_paired_difference_scenarios"]
    standardized = [abs(row["analytical_positive_rejection_power"] - row["simulated_positive_rejection_power"]) /
                    max(row["monte_carlo_standard_error"], 1e-12) for row in rows]
    assert max(standardized) < 5
    null = power["null_calibration"]
    assert all(abs(row["two_sided_null_rejection_rate"] - .05) < .012 for row in null)
    assert all(abs(row["positive_superiority_null_rate"] - .025) < .012 for row in null)
    # Exercise image-source and rendering code only with the existing development IDs.
    folder = root / "artifacts/notebook_12_validation"
    folder.mkdir(parents=True, exist_ok=True)
    dataset_manifest = _load12(root / "data/food101/dataset_manifest.json")
    gallery = _gallery12(root, folder, predictions, labels, ids, dataset_manifest["class_names"], 20261401)
    gallery["scope"] = "SMOKE_ONLY_REUSED_DEVELOPMENT_NOT_FRESH_VALIDATION"
    receipt = {"status": "PASS", "analysis_notebook_sha256": _sha12(_ANALYSIS_SOURCE12),
        "scope": "actual previously reported development predictions only; no fresh reserve or official test predictions consumed",
        "scipy_paired_t_agreement": True, "signflip_exact_count": result["signflip_enumerations"],
        "max_simulation_standardized_error": max(standardized), "bootstrap_reproducible": True,
        "development_test": result, "power_requirements": power["requirements_80_percent"],
        "null_calibration": null, "gallery_smoke": gallery}
    _write12(folder / "analysis_smoke.json", receipt)
    return {"status": "PASS", "receipt": str(folder / "analysis_smoke.json"),
            "analysis_notebook_sha256": receipt["analysis_notebook_sha256"], "gallery": gallery["path"]}
